<a href="https://colab.research.google.com/github/RB1Student/ISYS2001-Assignment-2-Project/blob/main/BankDash_Application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-genai

In [3]:
# =====================================================
#  BANK STATEMENT APP: ACCOUNT + DASHBOARD + CHATBOT
#  Paste this whole thing into ONE Colab cell and run it.
# =====================================================
import io
import html
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import userdata
from google import genai
from google.genai import types

# ---------- 0. Connect to Gemini ----------
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-3.5-flash"   # check AI Studio for the current model name


# ---------- 1. Loading + cleaning the CSV ----------
def read_uploaded_csv(upload_widget):
    """Read the file from the upload button (works with old and new ipywidgets)."""
    value = upload_widget.value
    if isinstance(value, dict):          # older ipywidgets
        filename = next(iter(value.keys()))
        file_info = value[filename]
    else:                                # newer ipywidgets
        file_info = value[0]
        filename = file_info["name"]
    df = pd.read_csv(io.BytesIO(bytes(file_info["content"])))
    return df, filename


def clean_money(series):
    return (
        series.astype(str)
        .str.replace(r"[^\d.\-]", "", regex=True)   # remove $, commas, spaces, "—"
        .replace("", "0")                           # empty strings become 0
        .astype(float)
    )


def clean_statement(df):
    for col in ["Income", "Expense", "Balance"]:
        df[col] = clean_money(df[col])
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
    df["Month"] = df["Date"].dt.strftime("%B %Y")   # e.g. "October 2026"
    return df


# ---------- 2. Dashboard pieces ----------
def kpi_card(title, value, colour="#1a237e"):
    return f"""
    <div style="flex:1; min-width:180px; background:#ffffff; border-radius:10px;
                padding:16px 20px; box-shadow:0 2px 6px rgba(0,0,0,0.15);
                border-top:5px solid {colour}; font-family:Arial, sans-serif;">
        <div style="font-size:13px; color:#666; text-transform:uppercase;">{title}</div>
        <div style="font-size:26px; font-weight:bold; color:{colour}; margin-top:6px;">{value}</div>
    </div>"""


def plot_charts(df, top_n=8):
    """Bar chart (monthly expense) and pie chart (expense by category) side by side."""
    fig, (ax_bar, ax_pie) = plt.subplots(
        1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 1.1]}
    )

    # --- Bar chart: monthly expense ---
    monthly_expense = df.groupby("Month", sort=False)["Expense"].sum()
    monthly_expense.plot(kind="bar", color="skyblue", ax=ax_bar)
    ax_bar.set_title("Monthly Expenses")
    ax_bar.set_xlabel("Month")
    ax_bar.set_ylabel("Total Expense ($)")
    ax_bar.tick_params(axis="x", rotation=45)

    # --- Pie chart: expense by category ---
    category_expense = (
        df.groupby("Name")["Expense"].sum()
        .loc[lambda s: s > 0]
        .sort_values(ascending=False)
    )

    if category_expense.empty:
        ax_pie.text(0.5, 0.5, "No expenses to show", ha="center", va="center")
        ax_pie.axis("off")
    else:
        if len(category_expense) > top_n:
            top = category_expense.iloc[:top_n]
            other = category_expense.iloc[top_n:].sum()
            category_expense = top.copy()
            category_expense["Other"] = other

        total = category_expense.sum()
        wedges, texts, autotexts = ax_pie.pie(
            category_expense,
            autopct="%1.1f%%",
            startangle=90,
            counterclock=False,
            pctdistance=0.8,
            wedgeprops={"edgecolor": "white"},
        )
        labels = [f"{name}  -  ${amt:,.2f}" for name, amt in category_expense.items()]
        ax_pie.legend(
            wedges,
            labels,
            title=f"Category (Total: ${total:,.2f})",
            loc="upper center",
            bbox_to_anchor=(0.5, -0.02),
            ncol=2,
            frameon=False,
        )
        ax_pie.set_title("Expenses by Category")

    plt.tight_layout()
    plt.show()


def display_financial_dashboard(df, user):
    """Joins the KPI cards, charts and transaction table into one dashboard."""
    balance = df["Balance"].iloc[-1]
    income = df["Income"].sum()
    expense = df["Expense"].sum()
    net = income - expense
    net_colour = "#2e7d32" if net >= 0 else "#c62828"

    display(HTML(f"""
    <div style="font-family:Arial, sans-serif; margin:10px 0 15px 0;">
        <h2 style="margin:0; color:#1a237e;">{html.escape(user)}'s Financial Dashboard</h2>
        <div style="color:#666; font-size:13px;">Showing {len(df)} transaction(s)</div>
    </div>
    <div style="display:flex; gap:15px; flex-wrap:wrap; margin-bottom:20px;">
        {kpi_card("Current Balance", f"${balance:,.2f}", "#1a237e")}
        {kpi_card("Total Income", f"${income:,.2f}", "#2e7d32")}
        {kpi_card("Total Expense", f"${expense:,.2f}", "#c62828")}
        {kpi_card("Net (Income - Expense)", f"${net:,.2f}", net_colour)}
    </div>
    """))

    plot_charts(df)

    display(HTML("<h3 style='font-family:Arial, sans-serif; color:#1a237e;'>Transactions</h3>"))
    display(df.drop(columns="Month").reset_index(drop=True))


def build_dashboard_screen(df, user):
    """Dropdown filters + dashboard output."""
    month_dd = widgets.Dropdown(
        options=["All"] + list(df["Month"].unique()), description="Month:"
    )
    category_dd = widgets.Dropdown(
        options=["All"] + sorted(df["Name"].unique()), description="Category:"
    )
    output = widgets.Output()

    def update(change=None):
        filtered = df.copy()
        if month_dd.value != "All":
            filtered = filtered[filtered["Month"] == month_dd.value]
        if category_dd.value != "All":
            filtered = filtered[filtered["Name"] == category_dd.value]

        with output:
            output.clear_output(wait=True)
            if filtered.empty:
                display(HTML(
                    "<div style='background-color:#fff3e0; padding:10px; border-left:5px solid #ff9800;"
                    "border-radius:5px; color:#e65100;'>No transactions match those filters. "
                    "Please adjust your selections.</div>"
                ))
                return
            display_financial_dashboard(filtered, user)

    month_dd.observe(update, names="value")
    category_dd.observe(update, names="value")
    update()   # show the dashboard straight away

    return widgets.VBox([widgets.HBox([month_dd, category_dd]), output])


# ---------- 3. Chatbot screen ----------
def build_chat_screen(df, user):
    chat = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            system_instruction=(
                f"You are a friendly finance assistant talking to {user}. "
                "Answer using only this bank statement:\n\n"
                + df.drop(columns="Month").to_csv(index=False)
            )
        ),
    )

    chat_area = widgets.Output()
    text_box = widgets.Text(
        placeholder="Ask about your spending...",
        layout=widgets.Layout(width="500px"),
    )
    send_btn = widgets.Button(description="Send", button_style="primary")

    def send(_):
        question = text_box.value
        if not question:
            return
        text_box.value = ""
        with chat_area:
            print(f"You: {question}")
            try:
                reply = chat.send_message(question).text
            except Exception as error:
                reply = f"Something went wrong talking to the API: {error}"
            print(f"Gemini: {reply}\n")

    send_btn.on_click(send)
    text_box.on_submit(send)  # press Enter to send

    return widgets.VBox([
        widgets.HTML(f"<h2 style='font-family:Arial, sans-serif; color:#1a237e;'>"
                     f"Hi {html.escape(user)}! Ask me about your bank statement</h2>"),
        chat_area,
        widgets.HBox([text_box, send_btn]),
    ])


# ---------- 4. Account screen (log in / log off) ----------
def message_box(text, colour="#e65100", background="#fff3e0"):
    return (f"<div style='background:{background}; padding:10px; border-left:5px solid {colour};"
            f"border-radius:5px; color:{colour}; font-family:Arial, sans-serif;'>{text}</div>")


def placeholder(text):
    return widgets.HTML(message_box(text, colour="#1a237e", background="#e8eaf6"))


account_box = widgets.VBox()
dashboard_box = widgets.VBox()
chat_box = widgets.VBox()


def show_login_screen():
    name_box = widgets.Text(placeholder="Your name", description="Name:")
    upload = widgets.FileUpload(accept=".csv", multiple=False, description="Upload CSV")
    start_btn = widgets.Button(description="Load my statement", button_style="success", icon="check")
    message = widgets.HTML()

    def start(_):
        name = name_box.value.strip()
        if not name:
            message.value = message_box("Please enter your name.")
            return
        if not upload.value:
            message.value = message_box("Please upload your bank statement CSV.")
            return
        try:
            df, filename = read_uploaded_csv(upload)
            df = clean_statement(df)
        except Exception as error:
            message.value = message_box(f"Couldn't read that file: {html.escape(str(error))}")
            return
        log_in(name, df, filename)

    start_btn.on_click(start)

    account_box.children = [
        widgets.HTML("<h2 style='font-family:Arial, sans-serif; color:#1a237e;'>Welcome! Let's get started</h2>"),
        name_box,
        upload,
        start_btn,
        message,
    ]
    waiting = "Please enter your name and upload your CSV on the 👤 Account tab first."
    dashboard_box.children = [placeholder(waiting)]
    chat_box.children = [placeholder(waiting)]


def log_in(name, df, filename):
    global User, Bank_Statement
    User = name
    Bank_Statement = df

    logoff_btn = widgets.Button(description="Log off", button_style="danger", icon="sign-out")
    logoff_btn.on_click(log_off)

    account_box.children = [
        widgets.HTML(message_box(
            f"Logged in as <b>{html.escape(name)}</b><br>"
            f"File: {html.escape(filename)} ({len(df)} transactions)",
            colour="#2e7d32", background="#e8f5e9",
        )),
        logoff_btn,
    ]
    dashboard_box.children = [build_dashboard_screen(df, name)]
    chat_box.children = [build_chat_screen(df, name)]
    tabs.selected_index = 1   # jump to the dashboard


def log_off(_):
    global User, Bank_Statement
    User = None
    Bank_Statement = None
    show_login_screen()       # wipes the dashboard, chat and upload
    tabs.selected_index = 0   # back to the account tab


# ---------- 5. Tabs ----------
tabs = widgets.Tab(children=[account_box, dashboard_box, chat_box])
tabs.set_title(0, "👤 Account")
tabs.set_title(1, "📊 Dashboard")
tabs.set_title(2, "💬 Chatbot")

show_login_screen()
display(tabs)